# Prototype Analysis on Sample

Dieses Notebook testet erste Analyseideen auf einem kleinen Sample der bereinigten Parking-Violations-Daten.

## Ziel

- processed Parquet-Daten aus HDFS laden
- kleines 1%-Sample erstellen
- Analysefragen testen
- häufigste Violation Codes untersuchen
- häufigste Vehicle Makes untersuchen
- zeitliche Muster nach Monat, Wochentag und Tageszeit prüfen
- beurteilen, welche Queries und Charts für die finale Analyse sinnvoll sind

Die finalen Analysen werden später in `src/4_Analysis` auf dem vollständigen Datensatz ausgeführt.

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, desc

spark = SparkSession.builder \
    .appName("BDLC_Parking_Violations_Prototype") \
    .master("spark://bdlc-012.bdlc.ls.eee.intern:7077") \
    .config("spark.executor.cores", "4") \
    .config("spark.executor.memory", "8g") \
    .config("spark.cores.max", "4") \
    .getOrCreate()

spark

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/09 20:50:39 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/05/09 20:50:40 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/05/09 20:50:40 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


26/05/09 20:50:55 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


In [11]:
processed_path = "hdfs:///parking_violations/processed/parking_violations_cleaned"

df = spark.read.parquet(processed_path)

df.groupBy("fiscal_year").count().orderBy("fiscal_year").show()

+-----------+--------+
|fiscal_year|   count|
+-----------+--------+
|       2023|21563238|
|       2024|16099641|
|       2025|16557773|
+-----------+--------+



In [12]:
sample_df = df.sample(fraction=0.01, seed=42)

sample_count = sample_df.count()
sample_count

542899

In [13]:
sample_df.groupBy("violation_code", "violation_description") \
    .count() \
    .orderBy(desc("count")) \
    .show(20, truncate=False)

+--------------+------------------------------+------+
|violation_code|violation_description         |count |
+--------------+------------------------------+------+
|36            |PHTO SCHOOL ZN SPEED VIOLATION|186027|
|21            |21-No Parking (street clean)  |46195 |
|38            |38-Failure to Dsplay Meter Rec|38110 |
|14            |14-No Standing                |25654 |
|7             |FAILURE TO STOP AT RED LIGHT  |23015 |
|5             |BUS LANE VIOLATION            |22837 |
|40            |40-Fire Hydrant               |19215 |
|21            |No Parking Street Cleaning    |16963 |
|71            |71A-Insp Sticker Expired (NYS)|16056 |
|20            |20A-No Parking (Non-COM)      |14303 |
|70            |70A-Reg. Sticker Expired (NYS)|10473 |
|37            |37-Expired Parking Meter      |8748  |
|31            |31-No Stand (Com. Mtr. Zone)  |8331  |
|69            |69-Fail to Dsp Prking Mtr Rcpt|7541  |
|19            |19-No Stand (bus stop)        |7311  |
|16       

In [14]:
sample_df.groupBy("violation_code") \
    .count() \
    .orderBy(desc("count")) \
    .show(20, truncate=False)

+--------------+------+
|violation_code|count |
+--------------+------+
|36            |186027|
|21            |63383 |
|38            |38170 |
|14            |27148 |
|7             |23015 |
|5             |22837 |
|40            |21455 |
|20            |20731 |
|71            |18989 |
|70            |12897 |
|46            |11156 |
|37            |8758  |
|31            |8359  |
|74            |7803  |
|19            |7760  |
|69            |7541  |
|16            |7355  |
|12            |5793  |
|43            |5325  |
|15            |4968  |
+--------------+------+
only showing top 20 rows



In [15]:
sample_df.groupBy("vehicle_make") \
    .count() \
    .orderBy(desc("count")) \
    .show(20, truncate=False)

+------------+-----+
|vehicle_make|count|
+------------+-----+
|HONDA       |64226|
|TOYOT       |63793|
|FORD        |50775|
|NISSA       |42733|
|CHEVR       |29175|
|ME/BE       |27911|
|BMW         |26856|
|JEEP        |24730|
|HYUND       |18727|
|LEXUS       |13624|
|FRUEH       |12010|
|ACURA       |11907|
|SUBAR       |11746|
|KIA         |11047|
|DODGE       |10802|
|AUDI        |10299|
|VOLKS       |9962 |
|MAZDA       |9817 |
|RAM         |8329 |
|INFIN       |8140 |
+------------+-----+
only showing top 20 rows



In [16]:
sample_df.groupBy("fiscal_year", "issue_month") \
    .count() \
    .orderBy("fiscal_year", "issue_month") \
    .show(50)

+-----------+-----------+-----+
|fiscal_year|issue_month|count|
+-----------+-----------+-----+
|       2023|          1|13261|
|       2023|          2|12577|
|       2023|          3|15127|
|       2023|          4|13935|
|       2023|          5|15142|
|       2023|          6|16408|
|       2023|          7|28695|
|       2023|          8|32466|
|       2023|          9|25002|
|       2023|         10|15202|
|       2023|         11|14729|
|       2023|         12|12532|
|       2024|          1|12349|
|       2024|          2|12607|
|       2024|          3|13500|
|       2024|          4|13149|
|       2024|          5|14277|
|       2024|          6|13240|
|       2024|          7|15106|
|       2024|          8|14742|
|       2024|          9|12778|
|       2024|         10|13983|
|       2024|         11|13812|
|       2024|         12|11720|
|       2025|          1|12106|
|       2025|          2|11699|
|       2025|          3|14445|
|       2025|          4|14895|
|       

In [17]:
sample_df.groupBy("issue_weekday") \
    .count() \
    .orderBy("issue_weekday") \
    .show()

+-------------+-----+
|issue_weekday|count|
+-------------+-----+
|            1|53459|
|            2|78482|
|            3|87611|
|            4|81453|
|            5|88069|
|            6|85547|
|            7|68278|
+-------------+-----+



In [18]:
sample_df.filter(col("violation_hour").isNotNull()) \
    .groupBy("violation_hour") \
    .count() \
    .orderBy("violation_hour") \
    .show(24)

+--------------+-----+
|violation_hour|count|
+--------------+-----+
|             0| 6746|
|             1| 7957|
|             2| 6288|
|             3| 5189|
|             4| 4893|
|             5| 7697|
|             6|16598|
|             7|29946|
|             8|46092|
|             9|47498|
|            10|38886|
|            11|47075|
|            12|44193|
|            13|42353|
|            14|38555|
|            15|32809|
|            16|26308|
|            17|22386|
|            18|16710|
|            19|12375|
|            20|12062|
|            21|10853|
|            22| 9623|
|            23| 8555|
+--------------+-----+



## Erkenntnisse aus dem Prototyping

Das 1%-Sample enthält 542'884 Zeilen und ist damit gross genug, um Analyseideen zu testen.

Getestete Analysefragen:

1. **Welche Violation Codes kommen am häufigsten vor?**  
   Diese Analyse ist sinnvoll. `violation_code = 36` ist im Sample mit Abstand am häufigsten. Für die finale Analyse sollte primär nach `violation_code` gruppiert werden, da einzelne Codes unterschiedliche Beschreibungen haben können.

2. **Welche Vehicle Makes erhalten am häufigsten Parking Violations?**  
   Diese Analyse ist sinnvoll. Im Sample sind `HONDA`, `TOYOT`, `FORD` und `NISSA` besonders häufig.

3. **Gibt es zeitliche Muster nach Monat?**  
   Diese Analyse ist sinnvoll. Besonders FY2023 zeigt auffällige Werte in den Monaten Juli, August und September.

4. **Gibt es zeitliche Muster nach Wochentag?**  
   Diese Analyse ist sinnvoll. Sonntag weist deutlich weniger Verstösse auf als die Werktage.

5. **Gibt es zeitliche Muster nach Tageszeit?**  
   Diese Analyse ist sinnvoll, weil `violation_time` im Pre-processing bereinigt und als `violation_hour` verfügbar gemacht wurde. Im Sample zeigen sich hohe Werte insbesondere zwischen ca. 08:00 und 14:00 Uhr. Für diese Analyse werden nur Datensätze mit `violation_hour IS NOT NULL` verwendet.

Die getesteten Analysen werden im nächsten Schritt in `src/4_Analysis` auf dem vollständigen Datensatz ausgeführt.